## 基于混合 GPD 的个股尾部风险动态配置策略

**目标：** 每个交易日，预测股票池里每只股票发生极端亏损的风险。风险低的股票多配，风险高的股票少配；当整个股票池的风险都偏高时，降低总仓位、多留现金。

整个流程分为四步：

```
13 个因子  ──神经网络──▶  5 个分布参数  ──公式──▶  风险评分 R  ──配置规则──▶  个股权重 + 现金比例
```

---

### 第 1 步：神经网络输出五个参数

每个交易日 $t$，对股票 $i$ 取 13 个因子 $X_{i,t}$：10 个原始因子，加上 3 个历史因子（worst252、e63、echo_q）。所有因子只用到 $t$ 日及之前的数据。神经网络据此输出五个参数：

| 参数 | 含义 | 是否随股票、日期变化 | 取值范围 |
|---|---|---|---|
| $\xi_b$, $\beta_b$ | **基础分量**的形状和尺度，描述"普通的"超阈值亏损 | 是 | $0.01 < \xi_b < 0.35$ |
| $\xi_h$, $\beta_h$ | **重尾分量**的形状和尺度，描述"非常极端"的亏损 | 否，所有股票共用同一组 | $0.35 < \xi_h < 0.80$ |
| $\pi$ | 重尾分量所占的权重 | 是 | $0.03 \le \pi < 1$ |

- **形状参数 $\xi$：** 越大，尾部越"厚"，越容易出现特别大的亏损。
- **尺度参数 $\beta$：** 越大，亏损的整体幅度越大。
- **$\pi$ 的下限 0.03：** 保证重尾分量永远不会被完全关掉，任何股票都保留一定的极端风险。

---

### 第 2 步：五个参数描述的亏损分布

记日亏损为 $L$（即负的日收益），$u$ 是模型训练时使用的阈值。模型只描述亏损已经超过 $u$ 的情况，超出部分记为 $y = L - u$。

$y$ 的分布是两个 GPD（广义帕累托分布）的混合。它的生存函数，也就是"$y$ 超过某个值的概率"，为

$$
S(y) = (1-\pi)\, S_{\mathrm{GPD}}(y;\ \xi_b, \beta_b) \;+\; \pi\, S_{\mathrm{GPD}}(y;\ \xi_h, \beta_h),
$$

其中单个 GPD 的生存函数是

$$
S_{\mathrm{GPD}}(y;\ \xi, \beta) = \left(1 + \frac{\xi\, y}{\beta}\right)^{-1/\xi}.
$$

**直观理解：** 亏损一旦超过 $u$，有 $1-\pi$ 的可能按"普通"分布继续发展，有 $\pi$ 的可能按"极端"分布发展。

---

### 第 3 步：风险评分 $R$

设灾难性亏损阈值 $d = 7\%$。令 $a = d - u$，它是灾难线在 $y$ 上对应的位置。风险评分定义为**超过灾难线之后的期望超额亏损**：

$$
R_{i,t} = \mathbb{E}\left[(y - a)^{+}\right]
= (1-\pi_{i,t})\, R_{\mathrm{GPD}}(\xi_b, \beta_b) \;+\; \pi_{i,t}\, R_{\mathrm{GPD}}(\xi_h, \beta_h),
$$

其中 $(x)^{+} = \max(x, 0)$。每个分量的贡献是

$$
R_{\mathrm{GPD}}(\xi, \beta)
= \left(1 + \frac{\xi\, a}{\beta}\right)^{-1/\xi}
\times \frac{\beta + \xi\, a}{1 - \xi}.
$$

这个公式由两个因子相乘：
- **第一个因子：** 超过灾难线的概率。
- **第二个因子：** 一旦超过，平均还会再亏多少（Expected Shortfall）。

所以 $R$ 越大，这只股票的灾难性亏损风险越高。

---

### 第 4 步：个股权重和总仓位

**(a) 相对权重：风险倒数加权。** 风险越低，权重越高：

$$
v_{i,t} = \frac{(R_{i,t} + \delta)^{-\gamma}}{\sum_{j=1}^{N} (R_{j,t} + \delta)^{-\gamma}}.
$$

- **$\gamma$：** 控制对风险差异的敏感程度。越大，越偏向低风险股票；$\gamma = 0$ 就是等权。
- **$\delta$：** 一个很小的正数，防止 $R$ 接近 0 时数值不稳定。

**(b) 满仓风险：** 如果按 $v$ 满仓买入，组合的风险是

$$
B_t = \sum_{i=1}^{N} v_{i,t}\, R_{i,t}.
$$

**(c) 总仓位：** 和预先设定的风险预算 $R_{\mathrm{target}}$ 比较：

$$
c_t = \min\left(1,\ \frac{R_{\mathrm{target}}}{B_t}\right).
$$

满仓风险没有超过预算就满仓；超过了就按比例减仓。

**(d) 最终持仓：**

$$
w_{i,t} = c_t\, v_{i,t}, \qquad w_{\mathrm{cash},t} = 1 - c_t.
$$

---

### 局限

模型描述的是"亏损已经超过 $u$"这一前提下的分布，没有预测亏损超过 $u$ 这件事本身发生的概率。因此 $R_{i,t}$ 是**条件**尾部风险评分，可以用来比较不同股票、不同日期的相对风险，但不是完整的"每日预期灾难损失"。

### 从两参数模型到五参数模型：策略改了什么

**一句话总结：** 只有风险评分 $R$ 的计算方法变了。拿到 $R$ 之后，权重和仓位的规则（第 4 步）完全不变。

---

#### 改动 1：$R$ 的公式

| | 风险评分 |
|---|---|
| 旧模型（2 个参数） | $R = R_{\mathrm{GPD}}(\xi, \beta)$ |
| 新模型（5 个参数） | $R = (1-\pi)\, R_{\mathrm{GPD}}(\xi_b, \beta_b) + \pi\, R_{\mathrm{GPD}}(\xi_h, \beta_h)$ |

**为什么可以直接按 $\pi$ 加权相加？** 期望超额亏损可以写成生存函数的积分：

$$
R = \mathbb{E}\left[(y - a)^{+}\right] = \int_a^{\infty} S(y)\, dy.
$$

混合分布的 $S(y)$ 本身就是两个分量按 $1-\pi$ 和 $\pi$ 的加权和，积分可以拆开，所以 $R$ 也是两个分量的 $R$ 的加权和。每个分量直接套用原来的闭式公式即可，不需要数值积分。

**两种看起来合理、但其实是错的做法：**
1. **把五个参数"平均"成一组 $(\xi, \beta)$，再代入旧公式。** 两个 GPD 混合以后不再是一个 GPD，不存在这样一组等价的参数。
2. **先算混合后的"超过概率"和混合后的"平均额外损失"，再相乘。** 混合后的平均额外损失应该按"各分量超过灾难线的概率"加权，而不是按 $\pi$ 加权，很容易算错。直接把两个分量的 $R$ 相加，就绕开了这个问题。

---

#### 改动 2：重尾分量对 $R$ 的影响比 $\pi$ 看起来大

$\xi_h$ 可以接近 0.8，这时"平均额外损失"里的 $\frac{1}{1-\xi_h}$ 能放大到 5 倍。所以即使 $\pi$ 只有百分之几，重尾分量也可能贡献 $R$ 的一大半。

又因为 $\xi_h$ 和 $\beta_h$ 是所有股票共用的常数，重尾那一项可以写成 $\pi_{i,t} \times K$，其中 $K$ 对所有股票、所有日期都相同。由此得到两个结论：
- **股票之间的风险排序** 主要由 $\pi$ 和基础分量的 $(\xi_b, \beta_b)$ 决定。$\pi$ 越大的股票，分到的权重越少。
- **每只股票的 $R$ 都有下限 $0.03K$**，因为 $\pi \ge 0.03$。$R$ 不会接近 0，所以 $\delta$ 基本不再起作用，也不会出现某只股票权重特别大的情况。

想看每只股票的 $R$ 里有多少来自重尾分量，可以调用 `gpd_mixture_allocation(..., return_details=True)`，查看返回的 `R_base` 和 `R_heavy`。

---

#### 改动 3：阈值 $u$ 从模型读取

旧函数里 `u=0.03` 是写死的默认值。现在 $u$ 从 `preprocess.json` 的 `threshold_u_loss` 读出（即 `predictor.u`），作为必填参数传入，保证和训练时用的阈值一致。同时要求 $u < d$，否则灾难线落在阈值以下，公式没有意义。

---

#### 改动 4：输入格式和参数检查

- **输入：** 从 `(n, 2)` 数组改为 `(n, 5)` 数组，或带五个参数列的 DataFrame。
- **检查：**
  - $0 < \xi_b, \xi_h < 1$：$\xi \ge 1$ 时期望亏损是无穷大。
  - $\beta_b, \beta_h > 0$。
  - $0 \le \pi \le 1$。

---

#### 如何确认改对了

下面的"公式自检"单元格做了两项检查，不需要模型和数据：
1. **闭式公式 vs 数值积分：** 两者的相对误差在 $10^{-9}$ 量级。
2. **退化检查：** 令 $\pi = 0$，混合分布退化成单个 GPD $(\xi_b, \beta_b)$，新函数的输出和原来的两参数函数完全相同。

---

#### 代码中对应的函数

| 函数 | 作用 |
|---|---|
| `gpd_expected_excess` | 单个 GPD 的 $R_{\mathrm{GPD}}(\xi, \beta)$ |
| `mixture_tail_risk` | 五参数混合分布的 $R$，同时返回两个分量各自的贡献 |
| `allocate_from_risk` | 第 4 步：$R \to v \to B \to c \to w$，和旧版本相同 |
| `gpd_mixture_allocation` | 上面几步合起来：五个参数 $\to$ 个股权重和现金比例 |

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F

# 模型文件夹：训练好的网络参数 gpd_model.pt 和预处理统计量 preprocess.json

#神经网络参数的训练结果储存在哪个文件夹的路径下，就把路径的名称存储在Path()中。目前我的电脑中是存储在了artifacts文件夹下
ARTIFACT_DIR = Path("artifacts")
PARAM_COLS = ["xi_b", "beta_b", "xi_h", "beta_h", "pi"]

In [ ]:
# ==========================================
# 2. 数据预处理（和训练时相同的变换，使用训练集统计量）
# ==========================================

def load_meta(artifact_dir=ARTIFACT_DIR):
    """preprocess.json: feature order, train median / mean / std, clip, threshold u, model settings."""
    return json.loads((Path(artifact_dir) / "preprocess.json").read_text())


def preprocess(X, meta):
    """Raw factors (n, 13) in meta["feature_names"] order -> model input (n, 13).

    1. fill missing values with the training-set median of each factor
    2. standardise with the training-set mean and std (z-score)
    3. clip to +/- clip_z (5)
    """
    X = np.asarray(X, dtype="float64")
    X = np.where(np.isnan(X), np.asarray(meta["feature_median_fill"]), X)
    Z = (X - np.asarray(meta["feature_mean"])) / np.asarray(meta["feature_std"])
    return np.clip(Z, -meta["clip_z"], meta["clip_z"]).astype("float32")


# ==========================================
# 3. 神经网络
# ==========================================

class KerasBatchNorm1d(nn.Module):
    def __init__(self, num_features, eps=1e-3, momentum=0.99):
        super().__init__()
        self.eps, self.momentum = eps, momentum
        self.weight = nn.Parameter(torch.ones(num_features))
        self.bias = nn.Parameter(torch.zeros(num_features))
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var", torch.ones(num_features))

    def forward(self, x):
        if self.training:
            mean, var = x.mean(dim=0), x.var(dim=0, unbiased=False)
            with torch.no_grad():
                self.running_mean.mul_(self.momentum).add_((1.0 - self.momentum) * mean)
                self.running_var.mul_(self.momentum).add_((1.0 - self.momentum) * var)
        else:
            mean, var = self.running_mean, self.running_var
        return (x - mean) / torch.sqrt(var + self.eps) * self.weight + self.bias

# Define the neural network model and forward propagation
class GateMixtureGPDNet(nn.Module):
    """forward(x) -> (B, 5) = [xi_b, beta_b, xi_h, beta_h, pi]; x is the preprocessed input."""

    def __init__(self, scale, input_dim=13, hidden_units=(224, 88), gate_units=32, pi_min=0.03):
        super().__init__()
        h1, h2 = hidden_units
        self.pi_min = pi_min
        self.trunk = nn.Sequential(
            nn.Linear(input_dim, h1, bias=False), KerasBatchNorm1d(h1), nn.ReLU(),
            nn.Linear(h1, h2, bias=False), KerasBatchNorm1d(h2), nn.ReLU(),
        )
        self.base_head = nn.Linear(h2, 2, bias=True)
        self.gate = nn.Sequential(
            nn.Linear(h2, gate_units, bias=False), KerasBatchNorm1d(gate_units), nn.ReLU(),
            nn.Linear(gate_units, 1, bias=True),
        )
        self.theta_xi = nn.Parameter(torch.zeros(()))
        self.theta_beta = nn.Parameter(torch.zeros(()))
        self.register_buffer("scale", torch.tensor(float(scale), dtype=torch.float32))

    #forward propagation of the model: given an input of standardized features (n=# of sample stocks, 13)
    #return the output matrix (n,5)
    def forward(self, x):
        h = self.trunk(x)
        a = self.base_head(h)
        xi_b = 0.01 + 0.34 * torch.sigmoid(a[:, 0])
        beta_b = self.scale * F.softplus(a[:, 1]) + 1e-6
        pi = self.pi_min + (1.0 - self.pi_min) * torch.sigmoid(self.gate(h).squeeze(-1))
        pi = pi.clamp(1e-6, 1 - 1e-6)
        ones = torch.ones_like(xi_b)
        xi_h = (0.35 + 0.45 * torch.sigmoid(self.theta_xi)) * ones
        beta_h = (self.scale * F.softplus(self.theta_beta) + 1e-6) * ones
        return torch.stack([xi_b, beta_b, xi_h, beta_h, pi], dim=1)


#This class combine the previous functions into a single class, which can be used to predict the GPD parameters for new data

class GPDPredictor:

    def __init__(self, artifact_dir=ARTIFACT_DIR):
        artifact_dir = Path(artifact_dir)
        self.meta = load_meta(artifact_dir)
        m = self.meta
        self.feature_names = m["feature_names"]
        self.u = float(m["threshold_u_loss"])
        self.model = GateMixtureGPDNet(m["gpd_scale"], input_dim=len(self.feature_names),
                                       hidden_units=tuple(m["hidden_units"]),
                                       gate_units=m["gate_units"], pi_min=m["pi_min"])
        state = torch.load(artifact_dir / m["model_file"], map_location="cpu", weights_only=True)
        self.model.load_state_dict(state, strict=True)
        self.model.eval()                       # BatchNorm uses the trained moving statistics

    def preprocess(self, X):
        """Section 2's preprocess() with this model's saved statistics."""
        return preprocess(X, self.meta)

    @torch.no_grad()
    def forward(self, X_raw, chunk_size=65536):
        """Raw factor matrix (n, 13) in feature_names order -> parameter array (n, 5).

        分块计算，回测时一次传入几十万行也不会占满内存。eval 模式下 BatchNorm 用的是
        训练好的固定统计量，所以分块与否结果完全相同。
        """
        x = self.preprocess(X_raw)
        out = [self.model(torch.from_numpy(x[i:i + chunk_size])).numpy()
               for i in range(0, len(x), chunk_size)]
        return np.concatenate(out).astype("float64")

    def predict(self, df):
        """DataFrame with the 13 factor columns -> copy with xi_b, beta_b, xi_h, beta_h, pi added."""
        missing = [c for c in self.feature_names if c not in df.columns]
        if missing:
            raise ValueError(f"missing factor columns: {missing}")
        P = self.forward(df[self.feature_names].values)
        out = df.copy()
        for i, c in enumerate(PARAM_COLS):
            out[c] = P[:, i]
        out["q99_loss"] = self.quantile(out, 0.99)
        return out

    @staticmethod
    def _params(params):
        return [np.asarray(params[c], dtype="float64") for c in PARAM_COLS]

    def survival(self, params, loss):
        """P(L > loss | L > u) for a loss level (e.g. 0.10 = 10%); loss <= u gives 1."""
        xi_b, beta_b, xi_h, beta_h, pi = self._params(params)
        y = np.maximum(np.asarray(loss, dtype="float64") - self.u, 0.0)
        sf = lambda xi, b: np.exp(-np.log1p(xi * y / b) / xi)
        return (1 - pi) * sf(xi_b, beta_b) + pi * sf(xi_h, beta_h)

    def quantile(self, params, q):
        """Loss level with P(L <= level | L > u) = q, solved by bisection (vectorised)."""
        xi_b, beta_b, xi_h, beta_h, pi = self._params(params)
        target = 1.0 - q
        qb = beta_b / xi_b * ((1 - q) ** (-xi_b) - 1)
        qh = beta_h / xi_h * ((1 - q) ** (-xi_h) - 1)
        lo, hi = np.zeros_like(qb), np.maximum(qb, qh)          # S(hi) <= 1 - q
        for _ in range(100):
            mid = 0.5 * (lo + hi)
            s = (1 - pi) * np.exp(-np.log1p(xi_b * mid / beta_b) / xi_b) \
                + pi * np.exp(-np.log1p(xi_h * mid / beta_h) / xi_h)
            lo, hi = np.where(s > target, mid, lo), np.where(s > target, hi, mid)
        return self.u + 0.5 * (lo + hi)

In [ ]:
# 策略函数


def gpd_expected_excess(xi, beta, a):
    """单个 GPD 分布的期望超额损失:  E[(Y - a)^+] = S(a) * (beta + xi*a) / (1 - xi)，要求 0 < xi < 1。

    S(a)                 = P(Y > a)：超过灾难线的概率
    (beta + xi*a)/(1-xi) = E[Y - a | Y > a]：一旦超过，平均还要再亏多少
    """

    survival = np.exp(-np.log1p(xi * a / beta) / xi)
    return survival * (beta + xi * a) / (1.0 - xi)


def mixture_tail_risk(gpd_params, d, u):

    """
    五参数混合 GPD 的风险评分: R = R = (1 - π) * 基础分量的风险 + π * 重尾分量的风险

    返回 (R, base_part, heavy_part)，后两个用于模型检验
    """
    P = _as_param_array(gpd_params)
    xi_b, beta_b, xi_h, beta_h, pi = P.T
    a = d - u
    base_part = (1.0 - pi) * gpd_expected_excess(xi_b, beta_b, a)
    heavy_part = pi * gpd_expected_excess(xi_h, beta_h, a)
    return base_part + heavy_part, base_part, heavy_part


def _as_param_array(gpd_params):
    """DataFrame（含 PARAM_COLS 列）或 (n, 5) 数组 -> (n, 5) 数组，并检查参数范围。"""
    if isinstance(gpd_params, pd.DataFrame):
        P = gpd_params[PARAM_COLS].to_numpy(dtype=float)
    else:
        P = np.asarray(gpd_params, dtype=float)
    if P.ndim != 2 or P.shape[1] != 5 or len(P) == 0:
        raise ValueError("gpd_params must have shape (n, 5): [xi_b, beta_b, xi_h, beta_h, pi].")
    if not np.all(np.isfinite(P)):
        raise ValueError("GPD parameters must be finite.")
    xi_b, beta_b, xi_h, beta_h, pi = P.T
    if np.any((xi_b <= 0) | (xi_b >= 1) | (xi_h <= 0) | (xi_h >= 1)):
        raise ValueError("Require 0 < xi_b, xi_h < 1 (otherwise the expected excess loss is infinite).")
    if np.any(beta_b <= 0) or np.any(beta_h <= 0):
        raise ValueError("Require beta_b, beta_h > 0.")
    if np.any((pi < 0) | (pi > 1)):
        raise ValueError("Require 0 <= pi <= 1.")
    return P


#输入同一天所有股票的风险评分R, 以及目标能够接受的最大风险R_target, 以及gamma,delta两个参数（用于计算风险的相对权重v）,输出每一支股票的权重weights，以及现金仓位cash

def allocate_from_risk(R, R_target, gamma=1.0, delta=1e-6):


    R = np.asarray(R, dtype=float)
    if R_target < 0 or gamma < 0 or delta <= 0:
        raise ValueError("Invalid risk settings.")

    #计算vi = (Ri+delta)^(-gamma) / sum((Ri+delta)^(-gamma))的时候，为了防止数值溢出，先去对数再取指数，得到相同计算结果

    scores = -gamma * np.log(R + delta)

    #归一化，每一个vi都减去vi的最大值，防止指数运算溢出
    scores = np.exp(scores - np.max(scores))
    v = scores / scores.sum()

    B = np.sum(v * R)

    c = min(1.0, R_target / B) if B > 0 else 1.0

    weights = c * v
    cash = 1.0 - weights.sum()
    return weights, cash, v, B, c

'''
将之前的函数组合起来，生成最终的策略函数

'''

def gpd_mixture_allocation(gpd_params, R_target, u, gamma=1.0, delta=1e-6, d=0.07,
                           return_details=False):
    """
    gpd_params: (n, 5) 数组或 DataFrame = [xi_b, beta_b, xi_h, beta_h, pi]
    R_target:   目标灾难风险预算
    u:          模型的 GPD 阈值（必须用 predictor.u，和训练时一致）
    gamma:      相对权重对风险差异的敏感度
    delta:      数值稳定用的小正数
    d:          灾难性亏损阈值（默认 7%,可以根据回测结果调整）

    返回 weights（占总资产的比例）, cash；return_details=True 时再返回一个诊断字典。
    """
    if not (0 <= u < d):
        raise ValueError("Invalid thresholds (need 0 <= u < d).")

    # 1. 混合 GPD 的灾难风险评分（这是唯一和两参数版本不同的一步）
    R, base_part, heavy_part = mixture_tail_risk(gpd_params, d, u)

    # 2-5. 和原来完全相同
    weights, cash, v, B, c = allocate_from_risk(R, R_target, gamma, delta)

    if return_details:
        details = dict(R=R, R_base=base_part, R_heavy=heavy_part, v=v, B=B, c=c)
        return weights, cash, details
    return weights, cash




### 回测：在 QuantConnect 上进行

这个 notebook 只负责说明策略和检查公式。回测放在 QuantConnect 上做，代码在 `quantconnect/` 文件夹里：

| 文件 | 作用 |
|---|---|
| `quantconnect/main.py` | QuantConnect 算法：每天计算 13 个因子 → 模型输出五个参数 → 本 notebook 中同样的策略公式 → 调仓 |
| `quantconnect/export_model_for_qc.py` | 在本地运行一次，把 `gpd_model.pt` 和 `preprocess.json` 转成 QuantConnect 能读取的 JSON 文件 |

`main.py` 里的策略函数（`gpd_expected_excess`、`mixture_tail_risk`、`allocate_from_risk`）和本 notebook 中的数学完全相同。如果在这里修改了策略公式，记得同步修改 `main.py`。